In [1]:
import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from dotenv import load_dotenv
from sklearn.metrics import f1_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from torch import nn
from transformers import (
    Trainer, AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments,
)

load_dotenv()

True

In [2]:
df = pd.read_csv("../../data/dontpatronizeme_pcl.tsv", sep="\t")
df["label_pcl"] = (df["label"] >= 2).astype(int)
len(df)

10469

In [3]:
def prepend(row):
    text = str(row["text"]).strip()
    return f"Keyword: {row["keyword"]}. Country: {row["country_code"]}. Text: {text}"


# Apply the function across the rows (axis=1)
df["cleaned_text"] = df.apply(prepend, axis=1)
df.head()

,par_id,art_id,keyword,country_code,text,label,label_pcl,cleaned_text
0,1,@@24942188,hopeless,ph,"We 're living in times of absolute insanity , ...",0,0,Keyword: hopeless. Country: ph. Text: We 're l...
1,2,@@21968160,migrant,gh,"In Libya today , there are countless number of...",0,0,Keyword: migrant. Country: gh. Text: In Libya ...
2,3,@@16584954,immigrant,ie,White House press secretary Sean Spicer said t...,0,0,Keyword: immigrant. Country: ie. Text: White H...
3,4,@@7811231,disabled,nz,Council customers only signs would be displaye...,0,0,Keyword: disabled. Country: nz. Text: Council ...
4,5,@@1494111,refugee,ca,""" Just like we received migrants fleeing El Sa...",0,0,"Keyword: refugee. Country: ca. Text: "" Just li..."


In [4]:
df_train_dev_labels = pd.read_csv("../../data/train_semeval_parids-labels.csv")
df_test_labels = pd.read_csv("../../data/dev_semeval_parids-labels.csv")
(
    len(df_train_dev_labels),
    len(df_test_labels),
    len(df_train_dev_labels) + len(df_test_labels)
)

(8375, 2094, 10469)

In [5]:
df_train_dev = df[df["par_id"].isin(df_train_dev_labels["par_id"])]
len(df_train_dev)

8375

In [6]:
df_test = df[df["par_id"].isin(df_test_labels["par_id"])]
len(df_test)

2094

In [7]:
df_train, df_dev = train_test_split(
    df_train_dev,
    test_size=0.1,
    random_state=42,
    shuffle=True,
    stratify=df_train_dev["label_pcl"],
)
len(df_train), len(df_dev)

(7537, 838)

In [9]:
from transformers import MarianMTModel, MarianTokenizer


def load_marian(src, tgt):
    name = f"Helsinki-NLP/opus-mt-{src}-{tgt}"
    print(f"Loading {name}...")
    tok = MarianTokenizer.from_pretrained(name)
    mdl = MarianMTModel.from_pretrained(name)
    return tok, mdl


def translate_batch(texts, tok, mdl, batch_size=16):
    results = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]
        inputs = tok(
            batch, return_tensors="pt", padding=True,
            truncation=True, max_length=512
        )
        generated = mdl.generate(**inputs, num_beams=4, max_length=512)
        results += [tok.decode(t, skip_special_tokens=True) for t in generated]
    return results


def backtranslate(texts, pivot_lang):
    """EN -> pivot_lang -> EN."""
    fwd_tok, fwd_mdl = load_marian("en", pivot_lang)
    bwd_tok, bwd_mdl = load_marian(pivot_lang, "en")
    pivot_texts = translate_batch(texts, fwd_tok, fwd_mdl)
    back_texts = translate_batch(pivot_texts, bwd_tok, bwd_mdl)
    # Free GPU memory between pivots
    del fwd_mdl, bwd_mdl
    if torch.mps.is_available():
        torch.mps.empty_cache()
    return back_texts


pcl_train = df_train[df_train["label_pcl"] == 1].copy()
raw_texts = pcl_train["text"].tolist()
print(f"Augmenting {len(pcl_train)} PCL examples via DE, FR, ES back-translation")

augmented_dfs = []
for pivot in ["de", "fr", "es"]:
    print(f"\n[{pivot.upper()}] EN -> {pivot} -> EN")
    aug_texts = backtranslate(raw_texts, pivot_lang=pivot)

    aug_df = pcl_train.copy()
    aug_df["text"] = aug_texts
    aug_df["cleaned_text"] = aug_df.apply(
        lambda r: f"Keyword: {r['keyword']}. Country: {r['country_code']}. Text: {r['text']}",
        axis=1
    )
    augmented_dfs.append(aug_df)

    print(f"  ORIGINAL:  {raw_texts[0][:120]}")
    print(f"  AUGMENTED: {aug_texts[0][:120]}")

df_train_aug = pd.concat(
    [df_train] + augmented_dfs, ignore_index=True
).sample(frac=1, random_state=42)

print(f"\nFinal train size: {len(df_train_aug):,}")
print(f"PCL% after augmentation: {df_train_aug.label_pcl.mean() * 100:.1f}%")
df_train_aug["label_pcl"].value_counts()

Augmenting 715 PCL examples via DE, FR, ES back-translation

[DE] EN -> de -> EN
Loading Helsinki-NLP/opus-mt-en-de...


Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


Loading Helsinki-NLP/opus-mt-de-en...


Loading weights:   0%|          | 0/258 [00:00<?, ?it/s]

...

The tied weights mapping and config for this model specifies to tie model.shared.weight to model.decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie model.shared.weight to model.encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/312M [00:00<?, ?B/s]

  ORIGINAL:  " Her Majesty the Queen 's Commonwealth Points of Light recognises Dr Madhusudhan as a role model of volunteerism . In p
  AUGMENTED: "His Majesty the points of light of the Queen's Commonwealth recognizes Dr. Madhusudhan as a role model of volunteerism.

Final train size: 9,682
PCL% after augmentation: 29.5%


label_pcl
0    6822
1    2860
Name: count, dtype: int64

In [10]:
df_train_aug["label_pcl"].value_counts(normalize=True)

label_pcl
0    0.704606
1    0.295394
Name: proportion, dtype: float64

In [11]:
df_dev["label_pcl"].value_counts(normalize=True)

label_pcl
0    0.905728
1    0.094272
Name: proportion, dtype: float64

In [12]:
tokenizer = AutoTokenizer.from_pretrained("roberta-base")


def tokenize_function(examples):
    return tokenizer(examples["cleaned_text"], padding="max_length", truncation=True, max_length=256)

In [13]:
hf_dataset_train = Dataset.from_pandas(df_train_aug)
tokenized_dataset_train = hf_dataset_train.map(tokenize_function, batched=True)
tokenized_dataset_train = tokenized_dataset_train.rename_column("label_pcl", "labels")
tokenized_dataset_train.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

Map:   0%|          | 0/9682 [00:00<?, ? examples/s]

In [14]:
hf_dataset_dev = Dataset.from_pandas(df_dev)
tokenized_dataset_dev = hf_dataset_dev.map(tokenize_function, batched=True)
tokenized_dataset_dev = tokenized_dataset_dev.rename_column("label_pcl", "labels")
tokenized_dataset_dev.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

Map:   0%|          | 0/838 [00:00<?, ? examples/s]

In [15]:
# Check for Apple Silicon (mps), then NVIDIA (cuda), then fall back to CPU
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
device

device(type='mps')

In [16]:
class_weights = compute_class_weight(
    "balanced",
    classes=np.unique(df_train_aug["label_pcl"]),
    y=df_train_aug["label_pcl"]
)

class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32).to(device)


# 2. Create a Custom Trainer to use the weights
class WeightedLossTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        # Forward pass
        outputs = model(**inputs)
        logits = outputs.logits
        # Apply the custom weights to the Cross Entropy Loss
        loss_fct = nn.CrossEntropyLoss(weight=class_weights_tensor)
        loss = loss_fct(logits.view(-1, self.model.config.num_labels), labels.view(-1))

        return (loss, outputs) if return_outputs else loss


class_weights_tensor

tensor([0.7096, 1.6927], device='mps:0')

In [17]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    f1 = f1_score(labels, preds, pos_label=1)
    return {"f1_pcl": f1}

In [19]:
model = AutoModelForSequenceClassification.from_pretrained("roberta-base", num_labels=2)

training_args = TrainingArguments(
    output_dir=".",
    eval_strategy="epoch",
    save_strategy="epoch",
    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=1e-5,
    warmup_ratio=0.1,
    weight_decay=0.1,
    load_best_model_at_end=True,
    metric_for_best_model="f1_pcl",
    greater_is_better=True,
    logging_steps=50,
)

trainer = WeightedLossTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset_train,
    eval_dataset=tokenized_dataset_dev,
    compute_metrics=compute_metrics,
)

trainer.train()

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
classifier.out_proj.bias        | MISSING    | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.dense.weight         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
/Users/henrytsyu/Imperial/70

Epoch,Training Loss,Validation Loss,F1 Pcl
1,0.243620,0.402162,0.529412
2,0.191842,0.551090,0.385965
3,0.139347,0.504987,0.554913
4,0.060129,0.700724,0.576471
5,0.086313,0.767408,0.557823


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/Users/henrytsyu/Imperial/70016-cw/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/Users/henrytsyu/Imperial/70016-cw/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

TrainOutput(global_step=3030, training_loss=0.1881577500022284, metrics={'train_runtime': 1727.4854, 'train_samples_per_second': 28.023, 'train_steps_per_second': 1.754, 'total_flos': 6368603094988800.0, 'train_loss': 0.1881577500022284, 'epoch': 5.0})

In [20]:
# 1. Get the model's predictions on your internal dev set
predictions = trainer.predict(tokenized_dataset_dev)

# 2. Convert raw logits into 0 or 1 labels
preds = np.argmax(predictions.predictions, axis=-1)

# 3. Get the true labels
labels = predictions.label_ids

# 4. Calculate the F1 score for the positive class (PCL = 1)
f1 = f1_score(labels, preds)
print("Internal Dev F1 Score:", f1)

/Users/henrytsyu/Imperial/70016-cw/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Internal Dev F1 Score: 0.5764705882352941


In [21]:
# This will show you exactly how many False Positives vs False Negatives you have
print(classification_report(labels, preds))

              precision    recall  f1-score   support

           0       0.96      0.94      0.95       759
           1       0.54      0.62      0.58        79

    accuracy                           0.91       838
   macro avg       0.75      0.78      0.76       838
weighted avg       0.92      0.91      0.92       838



In [22]:
trainer.save_model("./BestModel")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [23]:
tokenizer.save_pretrained("./BestModel")

('./BestModel/tokenizer_config.json', './BestModel/tokenizer.json')

In [24]:
# Create dev.txt

hf_dataset_test = Dataset.from_pandas(df_test)
tokenized_dataset_test = hf_dataset_test.map(tokenize_function, batched=True)
tokenized_dataset_test = tokenized_dataset_test.rename_column("label_pcl", "labels")
tokenized_dataset_test.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

# 1. Get the model's predictions on test set
predictions = trainer.predict(tokenized_dataset_test)

# 2. Convert raw logits into 0 or 1 labels
preds = np.argmax(predictions.predictions, axis=-1)

# 3. Get the true labels
labels = predictions.label_ids

# 4. Calculate the F1 score for the positive class (PCL = 1)
f1 = f1_score(labels, preds)
print("Test F1 Score:", f1)

Map:   0%|          | 0/2094 [00:00<?, ? examples/s]

/Users/henrytsyu/Imperial/70016-cw/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Test F1 Score: 0.5775656324582339


In [25]:
print(classification_report(labels, preds))

              precision    recall  f1-score   support

           0       0.96      0.95      0.95      1895
           1       0.55      0.61      0.58       199

    accuracy                           0.92      2094
   macro avg       0.75      0.78      0.77      2094
weighted avg       0.92      0.92      0.92      2094



In [27]:
# Output test set (official dev set) predictions to dev.txt
with open("../../dev.txt", "w") as f:
    for p in preds:
        f.write(f"{p}\n")

print(f"Saved {len(preds)} lines to dev.txt")

Saved 2094 lines to dev.txt


In [36]:
df_test_hidden = pd.read_csv("../../data/task4_test.tsv", sep="\t")
len(df_test_hidden)

3832

In [37]:
df_test_hidden.head()

,par_id,art_id,keyword,country_code,text
0,t_0,@@7258997,vulnerable,us,"In the meantime , conservatives are working to..."
1,t_1,@@16397324,women,pk,In most poor households with no education chil...
2,t_2,@@16257812,migrant,ca,The real question is not whether immigration i...
3,t_3,@@3509652,migrant,gb,"In total , the country 's immigrant population..."
4,t_4,@@477506,vulnerable,ca,"Members of the church , which is part of Ken C..."


In [38]:
df_test_hidden["cleaned_text"] = df_test_hidden.apply(
    lambda r: f"Keyword: {r['keyword']}. Country: {r['country_code']}. Text: {r['text']}",
    axis=1
)
df_test_hidden.head()

,par_id,art_id,keyword,country_code,text,cleaned_text
0,t_0,@@7258997,vulnerable,us,"In the meantime , conservatives are working to...",Keyword: vulnerable. Country: us. Text: In the...
1,t_1,@@16397324,women,pk,In most poor households with no education chil...,Keyword: women. Country: pk. Text: In most poo...
2,t_2,@@16257812,migrant,ca,The real question is not whether immigration i...,Keyword: migrant. Country: ca. Text: The real ...
3,t_3,@@3509652,migrant,gb,"In total , the country 's immigrant population...",Keyword: migrant. Country: gb. Text: In total ...
4,t_4,@@477506,vulnerable,ca,"Members of the church , which is part of Ken C...",Keyword: vulnerable. Country: ca. Text: Member...


In [39]:
# Create test.txt

hf_dataset_test_hidden = Dataset.from_pandas(df_test_hidden)
tokenized_dataset_test_hidden = hf_dataset_test_hidden.map(tokenize_function, batched=True)
tokenized_dataset_test_hidden.set_format("torch", columns=["input_ids", "attention_mask"])

# 1. Get the model's predictions on hidden test set
predictions = trainer.predict(tokenized_dataset_test_hidden)

# 2. Convert raw logits into 0 or 1 labels
preds = np.argmax(predictions.predictions, axis=-1)
len(preds)

Map:   0%|          | 0/3832 [00:00<?, ? examples/s]

/Users/henrytsyu/Imperial/70016-cw/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


3832

In [41]:
# Output hidden test set predictions to test.txt
with open("../../test.txt", "w") as f:
    for p in preds:
        f.write(f"{p}\n")

print(f"Saved {len(preds)} lines to test.txt")

Saved 3832 lines to test.txt


In [42]:
# ── Get predictions on the official dev set (df_test) ─────────────────────
predictions = trainer.predict(tokenized_dataset_test)
preds = np.argmax(predictions.predictions, axis=-1)
labels = predictions.label_ids
probs = torch.softmax(torch.tensor(predictions.predictions), dim=-1)[:, 1].numpy()

print(classification_report(labels, preds, target_names=["No PCL", "PCL"]))

# ── Attach predictions back to the original dataframe ─────────────────────
df_analysis = df_test.copy().reset_index(drop=True)
df_analysis["pred"] = preds
df_analysis["true"] = labels
df_analysis["pcl_prob"] = probs

# ── Segment into four groups ───────────────────────────────────────────────
TP = df_analysis[(df_analysis.true == 1) & (df_analysis.pred == 1)]  # correctly caught PCL
TN = df_analysis[(df_analysis.true == 0) & (df_analysis.pred == 0)]  # correctly dismissed
FN = df_analysis[(df_analysis.true == 1) & (df_analysis.pred == 0)]  # PCL the model missed
FP = df_analysis[(df_analysis.true == 0) & (df_analysis.pred == 1)]  # wrongly flagged


# ── Print most confident examples from each group ─────────────────────────
def show_examples(df, label, sort_col, ascending, n=5):
    print(f"\n{'=' * 70}")
    print(f"{label}  ({len(df)} total)")
    print('=' * 70)
    for _, row in df.sort_values(sort_col, ascending=ascending).head(n).iterrows():
        print(f"  [prob={row.pcl_prob:.3f}] [{row.keyword}]")
        print(f"  {str(row.text)[:300]}")
        print()


# Most confident TPs — what PCL patterns is the model good at?
show_examples(
    TP, "TRUE POSITIVES — PCL correctly detected",
    "pcl_prob", ascending=False
)

# Least confident FNs — PCL the model almost missed (borderline)
show_examples(
    FN, "FALSE NEGATIVES — PCL missed (sorted by confidence, lowest first)",
    "pcl_prob", ascending=True
)

# Most confident FPs — what non-PCL is the model most wrongly sure about?
show_examples(
    FP, "FALSE POSITIVES — non-PCL wrongly flagged (most confident first)",
    "pcl_prob", ascending=False
)

# ── Breakdown by keyword — where does the model struggle most? ─────────────
keyword_stats = df_analysis.groupby("keyword").apply(
    lambda g: pd.Series(
        {
            "n_pcl": (g.true == 1).sum(),
            "TP": ((g.true == 1) & (g.pred == 1)).sum(),
            "FN": ((g.true == 1) & (g.pred == 0)).sum(),
            "FP": ((g.true == 0) & (g.pred == 1)).sum(),
            "recall": ((g.true == 1) & (g.pred == 1)).sum() / max((g.true == 1).sum(), 1),
            "precision": ((g.true == 1) & (g.pred == 1)).sum() / max((g.pred == 1).sum(), 1),
        }
    )
).reset_index().sort_values("recall")

print("\n" + "=" * 70)
print("PCL RECALL AND PRECISION BY KEYWORD")
print("=" * 70)
print(keyword_stats.to_string(index=False))

# ── FP deep-dive — what words appear most in false positives? ─────────────
from collections import Counter
import re


def top_words(texts, n=20):
    stopwords = {
        "the", "a", "an", "is", "was", "are", "were", "and", "or", "of", "to",
        "in", "that", "it", "he", "she", "they", "said", "for", "on", "at", "with"
    }
    words = []
    for t in texts:
        words += [w for w in re.findall(r'\b[a-z]+\b', str(t).lower()) if w not in stopwords]
    return Counter(words).most_common(n)


print("\n" + "=" * 70)
print("TOP WORDS IN FALSE POSITIVES (non-PCL wrongly flagged)")
print("=" * 70)
print(top_words(FP["text"].tolist()))

print("\nTOP WORDS IN FALSE NEGATIVES (PCL the model missed)")
print("=" * 70)
print(top_words(FN["text"].tolist()))

/Users/henrytsyu/Imperial/70016-cw/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


              precision    recall  f1-score   support

      No PCL       0.96      0.95      0.95      1895
         PCL       0.55      0.61      0.58       199

    accuracy                           0.92      2094
   macro avg       0.75      0.78      0.77      2094
weighted avg       0.92      0.92      0.92      2094


TRUE POSITIVES — PCL correctly detected  (121 total)
  [prob=1.000] [hopeless]
  The rights and needs of hundreds of thousands of older and disabled people are being neglected and their difficulties left to worsen under a hopeless system of social care . But while the King 's Fund report says that older people are faring the worst from the poor state of social care , this ignore

  [prob=1.000] [homeless]
  With a mission to " strive every day to create a safe haven where homeless women and children find stability and access to the basic needs of life " , the Elfreeda Foundation launched its open shelter on the 11th of August 2017 . In attendance were dignitaries 